In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("tipo", "csv")
dbutils.widgets.text("file", "renovacion_prestamo_clientes_score_financiero")
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_project_ssm")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsinhueproject")

In [0]:
tipo = dbutils.widgets.get("tipo")
file = dbutils.widgets.get("file")
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/{file}.{tipo}"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
file_score_financiero_schema = StructType(fields=[
    StructField("ID", IntegerType(), False),
    StructField("LINEA_RENOVADO", IntegerType(), True),
    StructField("PLAZO_RENOVADO", IntegerType(), True),
    StructField("USO_LINEA_TOTAL_TC_T2", DoubleType(), True),
    StructField("USO_TRIM_LINEA_BBVA", DoubleType(), True),
    StructField("NR_ENTIDADES_TOTAL_T2", IntegerType(), True),
    StructField("DIFF_NRO_ENTIDA_TOTALES_T2_T12", IntegerType(), True),
    StructField("SDO_CONSUMO_T2", DoubleType(), True),
    StructField("Ahorro_Sldo_Bco_T1", IntegerType(), True),
    StructField("PConsumo_Sldo_Bco_T1", IntegerType(), True),
    StructField("SDO_BCO_tot_sm_pasivo_Bco_6M", DoubleType(), True),
    StructField("SUELDO_ESTIMADO", DoubleType(), True),
    StructField("CUBRIR_DEUDA_CONSUMO_SF_RENOVA_PLD", DoubleType(), True)
])

In [0]:
df_score_financiero = spark.read\
.option("sep", ";") \
.option('header', True)\
.schema(file_score_financiero_schema)\
.csv(ruta)

In [0]:
df_renamed_score_financiero = df_score_financiero \
    .withColumnRenamed("LINEA_RENOVADO", "Linea_Renovado") \
    .withColumnRenamed("PLAZO_RENOVADO", "Plazo_Renovado") \
    .withColumnRenamed("USO_LINEA_TOTAL_TC_T2", "Uso_Linea") \
    .withColumnRenamed("USO_TRIM_LINEA_BBVA", "Uso_TrimLinea") \
    .withColumnRenamed("NR_ENTIDADES_TOTAL_T2", "Nro_Entidades") \
    .withColumnRenamed("DIFF_NRO_ENTIDA_TOTALES_T2_T12", "Dif_Entidades") \
    .withColumnRenamed("SDO_CONSUMO_T2", "Saldo_Consumo") \
    .withColumnRenamed("Ahorro_Sldo_Bco_T1", "Ahorro") \
    .withColumnRenamed("PConsumo_Sldo_Bco_T1", "Prestamo_vigente") \
    .withColumnRenamed("SDO_BCO_tot_sm_pasivo_Bco_6M", "Promed_6Mdeuda") \
    .withColumnRenamed("CUBRIR_DEUDA_CONSUMO_SF_RENOVA_PLD", "Deuda_Cubierta")

In [0]:
df_renamed_score_financiero.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.{file}")